# 11 — Spark History UI through the private cluster IAP tunnel
Open the actual remote Spark History Server locally to review and capture job pages.

In [ ]:
from pathlib import Path
import json
import os
import re
import shlex
import subprocess
from IPython.display import display
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
EXPECTED_RUN_ID = "20261012-full-01"
runtime_path = REPO_ROOT / "artifacts" / "execution" / "runs" / EXPECTED_RUN_ID / "runtime.json"
runtime = json.loads(runtime_path.read_text())
PROJECT_ID, BUCKET, RUN_ID = runtime["project_id"], runtime["bucket"], runtime["run_id"]
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))
from rainfall.billing import validate_runtime_credit_gate
REGION, CLUSTER = "asia-southeast1", "dsa5208-rainfall"
if runtime["billing_account_id"] != "01DAD8-AE5F1A-E3910A" or runtime["credit_id"] != "3LU63BPGY77JFDQQ":
    raise RuntimeError("Unexpected billing/credit in runtime manifest.")
def sdk(args, *, as_json=False):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+", "[redacted identity]", result.stderr.strip()))
    return json.loads(result.stdout) if as_json else result.stdout
billing = sdk(["billing", "projects", "describe", PROJECT_ID, "--format=json"], as_json=True)
credit_gate_status = validate_runtime_credit_gate(
    runtime, project_id=PROJECT_ID, billing_info=billing
)
display(credit_gate_status)
cluster = sdk(["dataproc", "clusters", "describe", CLUSTER, f"--project={PROJECT_ID}", f"--region={REGION}", "--format=json"], as_json=True)
display({"state": cluster["status"]["state"], "component_urls": cluster["config"].get("endpointConfig", {}).get("httpPorts", {})})
if cluster["status"]["state"] != "RUNNING":
    raise RuntimeError("Cluster is not RUNNING; wait for provisioning and rerun this notebook.")
gce = cluster["config"]["gceClusterConfig"]
if gce.get("serviceAccount") != f"rainfall-vm@{PROJECT_ID}.iam.gserviceaccount.com" or not gce.get("internalIpOnly"):
    raise RuntimeError("Unexpected VM identity or public network configuration.")
ZONE = gce["zoneUri"].rsplit("/", 1)[-1]
MASTER = cluster["config"]["masterConfig"]["instanceNames"][0]
def ssh(command):
    return sdk(["compute", "ssh", MASTER, f"--zone={ZONE}", f"--project={PROJECT_ID}", "--tunnel-through-iap", "--quiet", f"--command={command}"])
print({"run_id": RUN_ID, "zone": ZONE, "master": MASTER})



In [2]:
import socket
import subprocess

history_address = ("127.0.0.1", 18080)
with socket.socket() as probe:
    tunnel_active = probe.connect_ex(history_address) == 0
if not tunnel_active:
    log_path = REPO_ROOT / "artifacts" / "execution" / "history-tunnel.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a") as log:
        tunnel = subprocess.Popen(["gcloud", "compute", "ssh", MASTER, f"--zone={ZONE}", f"--project={PROJECT_ID}", "--tunnel-through-iap", "--quiet", "--ssh-flag=-N", "--ssh-flag=-oExitOnForwardFailure=yes", "--ssh-flag=-L127.0.0.1:18080:localhost:18080"], stdout=log, stderr=log, start_new_session=True)
    (REPO_ROOT / "artifacts" / "execution" / "history-tunnel.pid").write_text(str(tunnel.pid))
    display({"tunnel_pid": tunnel.pid, "actual_remote_spark_history_ui": "http://127.0.0.1:18080"})
else:
    display({"tunnel": "already active", "actual_remote_spark_history_ui": "http://127.0.0.1:18080"})

{'tunnel': 'already active',
 'actual_remote_spark_history_ui': 'http://127.0.0.1:18080'}

In [3]:
# Forward the live Spark UI as well as History Server for the current application.
import socket
live_address = ("127.0.0.1", 38535)
with socket.socket() as probe:
    live_ui_active = probe.connect_ex(live_address) == 0
if not live_ui_active:
    live_log_path = REPO_ROOT / "artifacts" / "execution" / "live-spark-ui-tunnel.log"
    with live_log_path.open("a") as log:
        live_tunnel = subprocess.Popen(["gcloud", "compute", "ssh", MASTER, f"--zone={ZONE}", f"--project={PROJECT_ID}", "--tunnel-through-iap", "--quiet", "--ssh-flag=-N", "--ssh-flag=-oExitOnForwardFailure=yes", "--ssh-flag=-L127.0.0.1:38535:localhost:38535"], stdout=log, stderr=log, start_new_session=True)
    (REPO_ROOT / "artifacts" / "execution" / "live-spark-ui-tunnel.pid").write_text(str(live_tunnel.pid))
    display({"live_ui_tunnel_pid": live_tunnel.pid, "spark_ui": "http://127.0.0.1:38535"})
else:
    display({"live_ui_tunnel": "already active", "spark_ui": "http://127.0.0.1:38535"})


{'live_ui_tunnel_pid': 32541, 'spark_ui': 'http://127.0.0.1:38535'}

In [ ]:
# Read actual applications, jobs and job details from this Spark History Server.
import csv
import json
import urllib.request
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

history = "http://127.0.0.1:18080"
def get_json(path):
    with urllib.request.urlopen(history + path, timeout=30) as response:
        return json.load(response)

def timestamp(value):
    return datetime.strptime(value, "%Y-%m-%dT%H:%M:%S.%fGMT")

applications = get_json("/api/v1/applications")
records = []
for app in applications:
    application_id = app["id"]
    summary = get_json(f"/api/v1/applications/{application_id}/jobs")
    def detail(job):
        return get_json(f"/api/v1/applications/{application_id}/jobs/{job['jobId']}")
    with ThreadPoolExecutor(max_workers=8) as pool:
        details = list(pool.map(detail, summary))
    by_id = {job["jobId"]: job for job in summary}
    for info in details:
        records.append({"application": app, **by_id[info["jobId"]], **info})

evidence = REPO_ROOT / "report" / "evidence" / "spark-ui" / RUN_ID
evidence.mkdir(parents=True, exist_ok=True)
(evidence / "spark-history-api.json").write_text(json.dumps(records, indent=2) + "\n")

# API metadata is retained separately; only index.csv tracks screenshot completion.
job_fields = ["application_id", "action", "job_id", "status", "duration_seconds", "stage_ids", "completed_stages", "failed_stages", "tasks", "completed_tasks", "failed_tasks"]
job_csv = evidence / "jobs-api.csv"
with job_csv.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=job_fields)
    writer.writeheader()
    for record in records:
        submitted = timestamp(record["submissionTime"])
        completed = timestamp(record["completionTime"]) if record.get("completionTime") else None
        writer.writerow({
            "application_id": record["application"]["id"],
            "action": record.get("description") or "unlabelled_action",
            "job_id": record["jobId"],
            "status": record.get("status", "UNKNOWN"),
            "duration_seconds": (completed - submitted).total_seconds() if completed else "",
            "stage_ids": json.dumps(record.get("stageIds", [])),
            "completed_stages": record.get("numCompletedStages"),
            "failed_stages": record.get("numFailedStages"),
            "tasks": record.get("numTasks"),
            "completed_tasks": record.get("numCompletedTasks"),
            "failed_tasks": record.get("numFailedTasks"),
        })

index_path = evidence / "index.csv"
previous = {}
if index_path.exists():
    with index_path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        if {"application_id", "action", "summary_screenshot", "detail_screenshots"}.issubset(reader.fieldnames or []):
            previous = {(row["application_id"], row["action"]): row for row in reader}

groups = {}
for record in records:
    application_id = record["application"]["id"]
    action = record.get("description") or "unlabelled_action"
    groups.setdefault((application_id, action), []).append(record)

index_fields = ["run_id", "application_id", "action", "job_ids", "job_statuses", "job_details", "summary_screenshot", "detail_screenshots", "evidence_status"]
with index_path.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=index_fields)
    writer.writeheader()
    for (application_id, action), jobs in sorted(groups.items()):
        jobs = sorted(jobs, key=lambda job: job["jobId"])
        details = []
        for job in jobs:
            submitted = timestamp(job["submissionTime"])
            completed = timestamp(job["completionTime"]) if job.get("completionTime") else None
            details.append({
                "job_id": job["jobId"],
                "status": job.get("status", "UNKNOWN"),
                "duration_seconds": (completed - submitted).total_seconds() if completed else None,
                "stage_ids": job.get("stageIds", []),
                "completed_stages": job.get("numCompletedStages"),
                "failed_stages": job.get("numFailedStages"),
                "tasks": job.get("numTasks"),
                "completed_tasks": job.get("numCompletedTasks"),
                "failed_tasks": job.get("numFailedTasks"),
            })
        old = previous.get((application_id, action), {})
        detail_paths = {str(job["jobId"]): "PENDING_SCREENSHOT" for job in jobs}
        if old.get("detail_screenshots"):
            prior_paths = json.loads(old["detail_screenshots"])
            detail_paths.update({job_id: path for job_id, path in prior_paths.items() if path != "PENDING_SCREENSHOT"})
        statuses = sorted({job.get("status", "UNKNOWN") for job in jobs})
        summary_path = old.get("summary_screenshot") or "PENDING_SCREENSHOT"
        screenshots_pending = summary_path == "PENDING_SCREENSHOT" or any(path == "PENDING_SCREENSHOT" for path in detail_paths.values())
        writer.writerow({
            "run_id": RUN_ID,
            "application_id": application_id,
            "action": action,
            "job_ids": json.dumps([job["jobId"] for job in jobs]),
            "job_statuses": json.dumps(statuses),
            "job_details": json.dumps(details, separators=(",", ":")),
            "summary_screenshot": summary_path,
            "detail_screenshots": json.dumps(detail_paths, separators=(",", ":")),
            "evidence_status": "PENDING_SPARK_WEB_UI_SCREENSHOTS" if screenshots_pending else "SCREENSHOTS_CAPTURED",
        })

display({
    "applications": len(applications),
    "jobs": len(records),
    "states": {state: sum(job.get("status") == state for job in records) for state in sorted({job.get("status") for job in records})},
    "actions": {action: sum(job.get("description") == action for job in records) for action in sorted({job.get("description") for job in records})},
    "action_index_rows": len(groups),
    "api_metadata_csv": str(job_csv),
    "screenshot_index": str(index_path),
    "screenshots_are_not_complete_until_captured_from_spark_ui": True,
})


## Capture the actual Spark Web UI screenshots

This notebook reads Spark History API metadata and maintains the evidence index; it does not create screenshots. For each completed notebook action, open the application in the Spark Web UI or Spark History Server, review the Jobs summary, then open every Job detail page. Capture the actual rendered web pages as JPEG files under `report/evidence/spark-ui/<RUN_ID>/`. Record the relative paths in `summary_screenshot` and `detail_screenshot` in `index.csv`. Keep unfinished or unreachable pages marked `pending`, and record when a screenshot shows only a retained checkpoint rather than final status. Do not use API JSON, a generated mock-up, or a Dataproc/YARN page as a substitute.
